In [ ]:
# 计算文本的表征，存成文件
from transformers import AutoTokenizer,AutoModel
import torch
import numpy as np
from tqdm import tqdm
from datasets import load_dataset,Dataset

# model_name = "princeton-nlp/unsup-simcse-bert-base-uncased"
# output_file = f"draw-data/emb/simcse-bb-stsb-train.npy"

# model_name = "bert-base-uncased"
# output_file = f"draw-data/emb/bb-stsb-train.npy"

model_name = "sosuke/ease-bert-base-uncased"
output_file = f"draw-data/emb/ease-bb-stsb-train.npy"


# dataset_name = "wiki1m_for_simcse.txt"
# dataset = load_dataset("LyuShawn/Dataset-LyuCSE", data_files=dataset_name)
# dataset = dataset['train']

dataset_name = "mteb/stsbenchmark-sts"
dataset = load_dataset(dataset_name, split="train")
sent_list = dataset['sentence1'] + dataset['sentence2']
dataset = Dataset.from_dict({"text": sent_list})

# 采样
# dataset = dataset.shuffle(seed=42).select(range(1000))
max_seq_length = 32
bs = 1024    # 以bs为单位进行推理
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)

# 用于存储所有文本的表征
all_embeddings = []

def prepare(examples):

    return tokenizer(examples["text"], padding=False, truncation=True, max_length=max_seq_length)

dataset = dataset.map(prepare, batched=True)


for batch in tqdm(dataset.batch(bs)):
    input_ids = batch["input_ids"]
    attention_mask = batch["attention_mask"]

    # 对齐
    max_len = max([len(ids) for ids in input_ids])
    input_ids = [ids + [tokenizer.pad_token_id] * (max_len - len(ids)) for ids in input_ids]
    attention_mask = [mask + [0] * (max_len - len(mask)) for mask in attention_mask]

    input_ids = torch.tensor(input_ids, dtype=torch.long).to(device)
    attention_mask = torch.tensor(attention_mask, dtype=torch.long).to(device)

    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)

    last_hidden_state = outputs.last_hidden_state
    pooler_output =last_hidden_state[:,0,:]
    all_embeddings.append(pooler_output.cpu().numpy())

# 将所有的文本表征和成一个array
all_embeddings = np.concatenate(all_embeddings, axis=0)
np.save(output_file, all_embeddings)
print(f"Saved to {output_file}")

In [10]:
# 跨语言文本表征向量计算
from transformers import AutoTokenizer,AutoModel
import torch
import numpy as np
from tqdm import tqdm
from datasets import load_dataset,Dataset

model_name = "FacebookAI/xlm-roberta-large"
output_file = f"draw-data/emb/xlm-r-large-multi-stsb-train.npz"

# model_name = "sentence-transformers/LaBSE"
# output_file = f"draw-data/emb/labse-multi-stsb-train.npz"

dataset_name = "mteb/stsb_multi_mt"
# lang_list = ["en","de","es","fr","it","nl","pl","pt","ru","zh"]
lang_list = ["en","de","fr","ru","zh"]

sample_num = 256

lang_sent_dict = {}
for lang in lang_list:
    dataset = load_dataset(dataset_name, name=lang, split="train",trust_remote_code=True)
    # 每个都只要sentence1，并采样1000个
    # 相同种子采样完对应id仍然是并行语料
    dataset = dataset.shuffle(seed=918).select(range(sample_num))
    lang_sent_dict[lang] = dataset['sentence1']

dataset = Dataset.from_dict(lang_sent_dict)

max_seq_length = 32
bs = 256    # 以bs为单位进行推理
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)

# 用于存储所有文本的表征
lang_emb_dict = {}

def prepare(examples):
    return tokenizer(examples["text"], padding=False, truncation=True, max_length=max_seq_length)

for lang in tqdm(lang_list):
    lang_dataset = Dataset.from_dict({"text": lang_sent_dict[lang]})
    lang_dataset = lang_dataset.map(prepare, batched=True)
    lang_all_embs = []

    for batch in tqdm(lang_dataset.batch(bs)):
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]

        # 对齐
        max_len = max([len(ids) for ids in input_ids])
        input_ids = [ids + [tokenizer.pad_token_id] * (max_len - len(ids)) for ids in input_ids]
        attention_mask = [mask + [0] * (max_len - len(mask)) for mask in attention_mask]

        input_ids = torch.tensor(input_ids, dtype=torch.long).to(device)
        attention_mask = torch.tensor(attention_mask, dtype=torch.long).to(device)

        with torch.no_grad():
            outputs = model(input_ids, attention_mask=attention_mask)

        last_hidden_state = outputs.last_hidden_state
        pooler_output =last_hidden_state[:,0,:]
        lang_all_embs.append(pooler_output.cpu().numpy())
    lang_emb_dict[lang] = np.concatenate(lang_all_embs, axis=0)


np.savez(output_file, **lang_emb_dict)
print(f"Saved to {output_file}")

  0%|          | 0/5 [00:00<?, ?it/s]

Map:   0%|          | 0/256 [00:00<?, ? examples/s]

Batching examples:   0%|          | 0/256 [00:00<?, ? examples/s]

 20%|██        | 1/5 [00:00<00:02,  1.95it/s]

Map:   0%|          | 0/256 [00:00<?, ? examples/s]

...

 80%|████████  | 4/5 [00:02<00:00,  1.93it/s]

Map:   0%|          | 0/256 [00:00<?, ? examples/s]

Batching examples:   0%|          | 0/256 [00:00<?, ? examples/s]

100%|██████████| 5/5 [00:02<00:00,  1.94it/s]

Saved to draw-data/emb/xlm-r-large-multi-stsb-train.npz


In [ ]:
import spacy
from transformers import AutoTokenizer
import numpy as np
from tqdm import tqdm

data_dir = 'draw-data/'

nlp = spacy.load('en_core_web_sm')
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

# 计算原句子
input_file = '../data/wiki1m_for_simcse.txt'
with open(input_file, 'r', encoding='utf-8') as f:
    sent_list = f.read().splitlines()

# 将数据拆分成较小的批次
batch_size = 5000
docs = list(tqdm(nlp.pipe(sent_list, batch_size=batch_size, disable=["ner", "parser", "textcat"], n_process=10), total=len(sent_list),desc='计算句子长度'))
sent_l_list = [len(doc) for doc in docs]

sent_l_arr = np.array(sent_l_list)
np.save(data_dir + 'c4-句子长度数组.npy', sent_l_arr)

token_l_list = []

for sent in tqdm(sent_list, desc='计算句子token长度'):
    token_l_list.append(len(tokenizer.tokenize(sent)))

token_l_arr = np.array(token_l_list)
np.save(data_dir + 'c4-句子token长度数组.npy', token_l_arr)

In [ ]:
# title和abstract转储
# 计算title和abstract的长度
import os
import sys
import json
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
import numpy as np
from knowledge.backend import MySQLClient
from tqdm import tqdm
import spacy
from transformers import AutoTokenizer

data_dir = 'draw-data/'
if os.path.exists(data_dir) == False:
    os.makedirs(data_dir)

nlp = spacy.load('en_core_web_sm')
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

MySQL = MySQLClient()

title_list = []
abstrct_list = []
offset = 0
limit = 1000
print('开始加载数据')
while True:
    sent_list = MySQL.batch_get_page_info_title_abstract(offset,limit)
    offset+=limit
    if not sent_list:
        break
    for sent in sent_list:
        # title_list.append(sent[1])
        abstrct_list.append(sent[2])

    # 每10万打印一次
    if offset % 100000 == 0:
        print(f'已加载{offset}条数据')
# # 存下来
# with open(data_dir + 'c4-title.txt', 'w', encoding='utf-8') as f:
#     for sent in title_list:
#         if sent:
#             f.write(sent + '\n')
with open(data_dir + 'c4-abstract.json', 'w', encoding='utf-8') as f:
    json.dump(abstrct_list, f)
print('加载数据完成')

In [ ]:
# 计算title和abstract的长度
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
import numpy as np
from knowledge.backend import MySQLClient
from tqdm import tqdm
import spacy
from transformers import AutoTokenizer
import json

data_dir = 'draw-data/'
if os.path.exists(data_dir) == False:
    os.makedirs(data_dir)

nlp = spacy.load('en_core_web_sm')
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

print('开始加载数据')
# with open(data_dir + 'c4-title.txt', 'r', encoding='utf-8') as f:
#     title_list = f.read().splitlines()
with open(data_dir + 'c4-abstract.json', 'r', encoding='utf-8') as f:
    abstract_list = json.load(f)
print('加载数据完成')

# 计算title
batch_size = 5000
n_p = 10
# docs = list(tqdm(nlp.pipe(title_list, batch_size=batch_size, disable=["ner", "parser", "textcat"], n_process=n_p), total=len(title_list),desc='计算title句子长度'))
# title_l_list = [len(doc) for doc in docs]

# title_sent_l_arr = np.array(title_l_list)
# np.save(data_dir + 'c4-title句子长度数组.npy', title_sent_l_arr)

# title_token_l_list = []

# for sent in tqdm(title_list, desc='计算title句子token长度'):
#     title_token_l_list.append(len(tokenizer.tokenize(sent)))

# title_token_l_arr = np.array(title_token_l_list)
# np.save(data_dir + 'c4-title句子token长度数组.npy', title_token_l_arr)

# 计算abstract

docs = list(tqdm(nlp.pipe(abstract_list, batch_size=batch_size, disable=["ner", "parser", "textcat"], n_process=n_p), total=len(abstract_list),desc='计算abstract句子长度'))
abstract_l_list = [len(doc) for doc in docs]

abstract_sent_l_arr = np.array(abstract_l_list)
np.save(data_dir + 'c4-abstract句子长度数组.npy', abstract_sent_l_arr)

abstract_token_l_list = []

for sent in tqdm(abstract_list, desc='计算title句子token长度'):
    abstract_token_l_list.append(len(tokenizer.tokenize(sent)))

abstract_token_l_arr = np.array(abstract_token_l_list)
np.save(data_dir + 'c4-abstract句子token长度数组.npy', abstract_token_l_arr)